# HepatoXu

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.HepatoXu)

class HepatoXu(LinearReferenceClock):
    pass



In [3]:
model = pya.models.HepatoXu()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "hepatoxu"
model.metadata["data_type"] = "DNA methylation"  # Paper: The study developed circulating tumour-DNA methylation markers.
model.metadata["species"] = "Homo sapiens"  # Paper: Plasma samples came from human HCC patients and normal controls.
model.metadata["year"] = 2017
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Xu, R.-H., et al. “Circulating tumour DNA methylation markers for diagnosis and prognosis of hepatocellular carcinoma.” Nature Materials 16: 1155–1161 (2017)."
model.metadata["doi"] = "https://doi.org/10.1038/nmat4997"
model.metadata["notes"] = "Ten-marker plasma cfDNA methylation logistic model producing the combined HCC diagnosis score (cd-score); this packaged model does not implement the separate eight-marker prognosis score."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["plasma cell-free DNA"]  # Paper: The diagnostic dataset comprised plasma cfDNA from HCC patients and normal controls.
model.metadata["predicts"] = ["hepatocellular carcinoma"]  # Paper: The ten-marker model produced a combined diagnosis score designated cd-score.
model.metadata["training_target"] = ["hepatocellular carcinoma"]  # Paper: The logistic regression was fitted as a binary prediction of HCC versus normal plasma samples.
model.metadata["unit"] = ["unitless"]  # Paper: The packaged LinearReferenceClock returns the weighted linear score without a sigmoid transformation.
model.metadata["model_type"] = "feature-selected logistic regression"  # Paper: Ten overlapping markers from LASSO and random forest were used as covariates in logistic regression.
model.metadata["platform"] = ["targeted bisulfite sequencing"]  # Paper: Chinese plasma methylation values were obtained by targeted bisulfite sequencing using molecular inversion probes.
model.metadata["population"] = "adults"  # Paper: The 1,933-sample dataset was split 2:1; the training set had 1,275 samples from 715 HCC and 560 normal samples.
model.metadata["journal"] = "Nature Materials"
model.metadata["last_author"] = "Kang Zhang"
model.metadata["n_features"] = 10
model.metadata["citations"] = 884
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o coefficients.csv https://raw.githubusercontent.com/bio-learn/biolearn/180852e2bab473303cb85da627178b1695ee9d86/biolearn/data/HepatoXu.csv")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
mask = df['CpGmarker'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 'CoefficientTraining'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['CpGmarker'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['CoefficientTraining'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Xu, Ruo-Han, et al. "Circulating tumour DNA methylation markers '
             'for diagnosis and prognosis of hepatocellular carcinoma." Nature '
             'Materials 16.11 (2017): 1155-1161.',
 'clock_name': 'hepatoxu',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1038/nmat4997',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2017}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg10428836',
 'cg26668608',
 'cg25754195',
 'cg05205842',
 'cg11606215',
 'cg24067911',
 'cg18196829',
 'cg23211949',
 'cg17213048',
 'cg25459300']
base_model_features: None

%==================================== Model Details ====================================%
Model Structure:



## Normal feature ranges

In [ ]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

## Basic test

In [ ]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
